In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score,f1_score,recall_score,roc_auc_score,precision_score
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV

import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer



In [3]:
df=pd.read_csv('phishing_processed.csv')

In [4]:
df.shape

(11190, 16)

In [5]:
df.head()

,domain_age,phish_hints,ratio_extHyperlinks,ratio_intHyperlinks,safe_anchor,ratio_extRedirection,length_url,ratio_digits_url,length_words_raw,longest_words_raw,length_hostname,avg_word_path,domain_in_title,char_repeat,status,status-encoded
0,5075.0,0,0.470588,0.529412,0.0,0.875000,37,0.000000,4,11,19,4.500000,0,4,legitimate,0
1,5767.0,0,0.033333,0.966667,100.0,0.000000,77,0.220779,4,32,23,14.666667,1,4,phishing,1
2,4004.0,0,0.000000,1.000000,100.0,0.000000,126,0.150794,12,17,50,8.142857,1,2,phishing,1
3,5075.0,0,0.026846,0.973154,62.5,0.250000,18,0.000000,1,5,11,0.000000,1,0,legitimate,0
4,8175.0,0,0.529412,0.470588,0.0,0.537037,55,0.000000,6,11,15,7.000000,0,3,legitimate,0


In [47]:
X=df.iloc[:,0:-2]
y=df.iloc[:,-1]

In [48]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [43]:
print(X_train.shape)
print(X_test.shape)

(8952, 14)
(2238, 14)


In [44]:
rf=RandomForestClassifier(n_estimators=400,max_features=0.75,max_samples=0.5,random_state=42,n_jobs=-1)



In [45]:
rf.fit(X_train,y_train)

,n_estimators,400
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.75
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
y_pred=rf.predict(X_test)

In [18]:
print("Accuracy :",accuracy_score(y_test,y_pred))
print("Precision:",precision_score(y_test,y_pred))
print("Recall:",recall_score(y_test,y_pred))
print("F1 score:",f1_score(y_test,y_pred))

Accuracy : 0.920017873100983
Precision: 0.9051959890610757
Recall: 0.9297752808988764
F1 score: 0.9173210161662817


- so accuracy is 92% means 92% prediction is correct

- precision is 90% means the out of all websites predicted as phishing , 90% were phishing

- Recall is 92% means the out of actual phishing url how many it catches is 92%

- F1 score is combination of precision and recall which is 91%


### Classification-report

In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1170
           1       0.91      0.93      0.92      1068

    accuracy                           0.92      2238
   macro avg       0.92      0.92      0.92      2238
weighted avg       0.92      0.92      0.92      2238



### Hyperparameter Tuning
#### (RandomSearchCV)

In [20]:
n_estimators=[20,60,100,200]
max_features=[0.25,0.5,0.75]
max_samples=[0.2,0.5,0.75]
max_depth=[2,8,None]

min_samples_leaf=[1,2]
min_samples_split=[2,5]


In [21]:
params={
    'n_estimators':n_estimators,
    'max_features':max_features,
    'max_samples':max_samples,
    'max_depth':max_depth,
    'min_samples_split':min_samples_split,
    'min_samples_leaf':min_samples_leaf
}

In [23]:
rf_random_search.best_params_

{'n_estimators': 60,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_samples': 0.75,
 'max_features': 0.5,
 'max_depth': None}

In [24]:
rf_random_search.best_score_

np.float64(0.918677371962232)

In [25]:
best_rf=rf_random_search.best_estimator_

##Saving best model

In [ ]:
# joblib.dump(
#     best_rf,
#     "random_forest_phishing.pkl"
# )

['random_forest_phishing.pkl']

In [30]:
rf_best = joblib.load("random_forest_phishing.pkl")

c:\Users\Dell\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Dell\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [31]:
y_pred = rf_best.predict(X_test)


In [32]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))


Accuracy : 0.920017873100983
Precision: 0.9029918404351768
Recall   : 0.9325842696629213
F1 Score : 0.9175495163519115


- The baseline Random Forest model and the hyperparameter-tuned Random Forest model achieved very similar performance.

- The tuned model obtained a slightly higher recall and F1-score, while the baseline model achieved marginally higher precision.


- Since phishing detection prioritizes minimizing false negatives and maximizing phishing detection rates, the tuned Random Forest model obtained through RandomizedSearchCV was selected as the final model for deployment.

### Logistic Regression

In [49]:
lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [50]:
lr_pipeline.fit(X_train,y_train)

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [65]:
print(lr_pipeline.named_steps)
print(lr_pipeline.named_steps["imputer"].statistics_)

{'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler(), 'model': LogisticRegression(max_iter=1000, random_state=42)}
[5.07500000e+03 0.00000000e+00 1.31178269e-01 7.50000000e-01
 2.50000000e+01 0.00000000e+00 4.70000000e+01 0.00000000e+00
 5.00000000e+00 1.10000000e+01 1.90000000e+01 4.87500000e+00
 1.00000000e+00 3.00000000e+00]


In [51]:
y_pred_lr=lr_pipeline.predict(X_test)
y_proba_lr=lr_pipeline.predict_proba(X_test)[:,1]

In [52]:
print("Logistic Regression")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1 Score :", f1_score(y_test, y_pred_lr))

Logistic Regression
Accuracy : 0.8266309204647007
Precision: 0.8366336633663366
Recall   : 0.7911985018726592
F1 Score : 0.8132820019249278


In [53]:
from sklearn.model_selection import cross_val_score

print(cross_val_score(lr_pipeline,X_train,y_train,cv=5,scoring="accuracy").mean())

0.8293121722828918


### KNN

In [54]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

In [55]:
knn_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        weights='distance'
    ))
])

In [56]:
knn_pipeline.fit(X_train,y_train)

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,feature_range,"(0, ...)"


In [57]:
y_pred_knn=knn_pipeline.predict(X_test)
y_proba_knn=knn_pipeline.predict_proba(X_test)[:,1]

In [58]:
print("K-nearest neighbors")
print("Accuracy :", accuracy_score(y_test, y_pred_knn))
print("Precision:", precision_score(y_test, y_pred_knn))
print("Recall   :", recall_score(y_test, y_pred_knn))
print("F1 Score :", f1_score(y_test, y_pred_knn))

K-nearest neighbors
Accuracy : 0.8941018766756033
Precision: 0.8938388625592417
Recall   : 0.8829588014981273
F1 Score : 0.8883655204898728


In [59]:
params={
    'model__n_neighbors':[3,5,10,15]
}

In [60]:
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(knn_pipeline,params,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)

,estimator,Pipeline(step...'distance'))])
,param_grid,"{'model__n_neighbors': [3, 5, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,missing_values,nan


In [61]:
grid.best_score_

np.float64(0.8827073293219667)

In [62]:
import pickle
pickle.dump(lr_pipeline,open('pipe_lr.pkl','wb'))

In [63]:
pickle.dump(knn_pipeline,open('pipe_knn.pkl','wb'))